# Advanced Statistical Analysis for Comparative Dashboard Exports

**Purpose:** This notebook provides advanced statistical analysis and publication-ready visualizations for Phase 3 comparative dashboard exports.

## When to Use This Tool

**Use this notebook when:**
- Comparing **5+ systems** requiring rigorous statistical analysis
- Preparing results for **academic publication** or **conference presentations**
- Need **effect size calculations** (Cohen's d) for pairwise comparisons
- Require **correlation analysis** between evaluation criteria
- Want **hierarchical clustering** to identify system groupings
- Need **high-resolution exports** (300 DPI) for publication
- Generating **supplementary materials** (LaTeX tables, markdown reports)

**Use Phase 3 Dashboard instead when:**
- Comparing **2-4 systems** for exploratory analysis
- Performing **initial evaluations** or **pilot studies**
- Need **quick visual comparisons** without statistical rigor
- Working in **early research phases** before formal publication

## Important Note on Statistical Testing

Traditional ANOVA and similar tests require **multiple replications** per system (e.g., 3+ independent evaluations). Since our framework typically involves **single scores per system** from synthesized judgments, we focus on:
- **Descriptive statistics** (mean, std dev, range)
- **Effect size measures** (Cohen's d) for practical significance
- **Correlation analysis** to understand criterion relationships
- **Clustering techniques** to reveal system groupings

For formal hypothesis testing, consider collecting multiple independent evaluations per system.

---

## Setup and Configuration

In [ ]:
# Cell 1: Import Required Libraries

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")
print(f"Analysis timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Cell 2: Configure Paths and Settings

# Base directories
BASE_DIR = Path('/workspace/aimusic-eval/phase1')
OUTPUTS_DIR = BASE_DIR / 'outputs'
VIZ_DIR = OUTPUTS_DIR / 'visualizations'
SUPP_DIR = OUTPUTS_DIR / 'supplementary'

# Create output directories
VIZ_DIR.mkdir(parents=True, exist_ok=True)
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
CONFIG = {
    'dpi': 300,
    'fig_width': 12,
    'fig_height': 8,
    'color_palette': 'viridis',
    'alpha': 0.05,  # Significance level for statistical tests
}

# Standard evaluation criteria (Phase 3 framework)
CRITERIA = [
    'usability',
    'generation_speed',
    'audio_quality',
    'stylistic_accuracy',
    'parameter_control',
    'content_generation_control',
    'daw_integration',
    'creative_workflow_fit'
]

CRITERIA_DISPLAY = {
    'usability': 'Usability',
    'generation_speed': 'Generation Speed',
    'audio_quality': 'Audio Quality',
    'stylistic_accuracy': 'Stylistic Accuracy',
    'parameter_control': 'Parameter Control',
    'content_generation_control': 'Content Generation Control',
    'daw_integration': 'DAW Integration',
    'creative_workflow_fit': 'Creative Workflow Fit'
}

print(f"✓ Paths configured")
print(f"  Outputs: {OUTPUTS_DIR}")
print(f"  Visualizations: {VIZ_DIR}")
print(f"  Supplementary: {SUPP_DIR}")

---

## Data Loading and Preparation

In [ ]:
# Cell 3: Load Comparative Analysis File

def load_comparative_analysis(filepath):
    """
    Load comparative analysis JSON file from Phase 3 dashboard export.
    
    Args:
        filepath: Path to comparative_analysis_*.json file
        
    Returns:
        dict: Parsed JSON data
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    # Validate structure
    required_keys = ['score_matrix', 'systems_compared', 'comparison_metadata']
    for key in required_keys:
        if key not in data:
            raise ValueError(f"Missing required key: {key}")
    
    return data

def list_available_files():
    """List all comparative_analysis_*.json files in outputs directory."""
    pattern = 'comparative_analysis_*.json'
    files = sorted(OUTPUTS_DIR.glob(pattern))
    return files

# List available files
available_files = list_available_files()
print(f"Found {len(available_files)} comparative analysis file(s):")
for i, f in enumerate(available_files, 1):
    print(f"  [{i}] {f.name}")

# Load the most recent file (or specify manually)
if available_files:
    # Use test file if available, otherwise most recent
    test_file = OUTPUTS_DIR / 'comparative_analysis_test_20251021.json'
    if test_file.exists():
        selected_file = test_file
        print(f"\n✓ Loading test file: {selected_file.name}")
    else:
        selected_file = available_files[-1]
        print(f"\n✓ Loading most recent file: {selected_file.name}")
    
    data = load_comparative_analysis(selected_file)
    print(f"  Systems compared: {data['comparison_metadata']['num_systems']}")
    print(f"  Evaluator: {data['comparison_metadata']['evaluator']}")
    print(f"  Date: {data['comparison_metadata']['comparison_date']}")
else:
    print("\n⚠ No files found. Please export from Phase 3 dashboard first.")
    data = None

In [ ]:
# Cell 4: Extract Score Matrix and System Metadata

def extract_score_matrix(data):
    """
    Extract score matrix as pandas DataFrame.
    
    Returns:
        DataFrame: Systems as rows, criteria as columns
    """
    score_matrix = data['score_matrix']['criteria_scores']
    system_names = [s['system_name'] for s in data['systems_compared']]
    
    # Create DataFrame (transpose so systems are rows)
    df = pd.DataFrame(score_matrix, index=CRITERIA).T
    df.index = system_names
    
    return df

def extract_system_metadata(data):
    """
    Extract system metadata as DataFrame.
    """
    systems = data['systems_compared']
    df = pd.DataFrame(systems)
    return df

if data:
    # Extract score matrix
    scores_df = extract_score_matrix(data)
    systems_df = extract_system_metadata(data)
    
    print("Score Matrix:")
    print(scores_df)
    print(f"\nShape: {scores_df.shape[0]} systems × {scores_df.shape[1]} criteria")
    print(f"\nSystem Metadata:")
    print(systems_df[['system_name', 'system_version', 'evaluation_date']])

---

## Descriptive Statistics

Comprehensive summary statistics provide the foundation for understanding system performance across criteria.

In [ ]:
# Cell 5: Compute Comprehensive Descriptive Statistics

def compute_descriptive_stats(scores_df):
    """
    Compute comprehensive descriptive statistics.
    
    Returns:
        DataFrame: Statistics per criterion and per system
    """
    stats_dict = {
        'Per-Criterion Statistics': scores_df.describe().T,
        'Per-System Statistics': scores_df.T.describe().T
    }
    
    # Add range and IQR
    stats_dict['Per-Criterion Statistics']['range'] = (
        scores_df.max() - scores_df.min()
    )
    stats_dict['Per-Criterion Statistics']['iqr'] = (
        scores_df.quantile(0.75) - scores_df.quantile(0.25)
    )
    
    stats_dict['Per-System Statistics']['range'] = (
        scores_df.T.max() - scores_df.T.min()
    )
    stats_dict['Per-System Statistics']['iqr'] = (
        scores_df.T.quantile(0.75) - scores_df.T.quantile(0.25)
    )
    
    return stats_dict

if data:
    stats = compute_descriptive_stats(scores_df)
    
    print("=" * 80)
    print("DESCRIPTIVE STATISTICS: Per-Criterion Analysis")
    print("=" * 80)
    print("\nInterpretation Guide:")
    print("  • Mean: Average score across all systems for each criterion")
    print("  • Std: Spread of scores (higher = more variation between systems)")
    print("  • Range: Difference between highest and lowest scoring systems")
    print("  • IQR: Middle 50% spread (robust to outliers)\n")
    
    display_cols = ['mean', 'std', 'min', 'max', 'range', 'iqr']
    print(stats['Per-Criterion Statistics'][display_cols].round(2))
    
    print("\n" + "=" * 80)
    print("DESCRIPTIVE STATISTICS: Per-System Analysis")
    print("=" * 80)
    print("\nInterpretation Guide:")
    print("  • Mean: Average score across all criteria for each system")
    print("  • Std: Consistency of system (lower = more balanced performance)")
    print("  • Range: Performance variability (min to max criterion score)\n")
    
    print(stats['Per-System Statistics'][display_cols].round(2))

---

## Effect Size Analysis

**Cohen's d** measures the magnitude of difference between two systems, independent of sample size. This provides **practical significance** beyond statistical significance.

### Interpretation Guide:
- **|d| < 0.2**: Negligible difference
- **0.2 ≤ |d| < 0.5**: Small effect
- **0.5 ≤ |d| < 0.8**: Medium effect  
- **|d| ≥ 0.8**: Large effect

Positive d means System A outperforms System B; negative means opposite.

In [ ]:
# Cell 6: Calculate Cohen's d for All Pairwise Comparisons

def cohens_d(group1, group2):
    """
    Calculate Cohen's d effect size.
    
    Args:
        group1, group2: Arrays of scores
        
    Returns:
        float: Cohen's d value
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    return (np.mean(group1) - np.mean(group2)) / pooled_std

def interpret_cohens_d(d):
    """Interpret Cohen's d magnitude."""
    abs_d = abs(d)
    if abs_d < 0.2:
        return 'negligible'
    elif abs_d < 0.5:
        return 'small'
    elif abs_d < 0.8:
        return 'medium'
    else:
        return 'large'

def pairwise_effect_sizes(scores_df):
    """
    Calculate Cohen's d for all pairwise system comparisons.
    
    Returns:
        DataFrame: Effect sizes (rows=comparisons, cols=criteria)
    """
    systems = scores_df.index.tolist()
    comparisons = []
    effect_sizes = []
    
    for i, sys1 in enumerate(systems):
        for sys2 in systems[i+1:]:
            comparison = f"{sys1} vs {sys2}"
            comparisons.append(comparison)
            
            row = []
            for criterion in scores_df.columns:
                # Get scores (single values per system)
                score1 = scores_df.loc[sys1, criterion]
                score2 = scores_df.loc[sys2, criterion]
                
                # For single values, use simple difference normalized by pooled SD
                # (requires multiple observations for true Cohen's d)
                diff = score1 - score2
                row.append(diff)
            
            effect_sizes.append(row)
    
    df = pd.DataFrame(effect_sizes, index=comparisons, columns=scores_df.columns)
    return df

if data:
    effect_sizes_df = pairwise_effect_sizes(scores_df)
    
    print("=" * 80)
    print("EFFECT SIZE ANALYSIS: Pairwise System Comparisons")
    print("=" * 80)
    print("\n⚠ Note: True Cohen's d requires multiple observations per system.")
    print("Values shown are raw score differences (System A - System B).\n")
    print("Interpretation:")
    print("  • Positive: First system scores higher")
    print("  • Negative: Second system scores higher")
    print("  • Magnitude: Size of performance difference\n")
    
    print(effect_sizes_df.round(2))
    
    # Identify largest differences
    print("\n" + "-" * 80)
    print("Largest Performance Differences (|difference| > 1.0):")
    print("-" * 80)
    
    for comparison in effect_sizes_df.index:
        for criterion in effect_sizes_df.columns:
            diff = effect_sizes_df.loc[comparison, criterion]
            if abs(diff) > 1.0:
                direction = "outperforms" if diff > 0 else "underperforms"
                sys1, sys2 = comparison.split(' vs ')
                print(f"  • {criterion}: {sys1} {direction} {sys2} by {abs(diff):.2f} points")

---

## Correlation Analysis

**Pearson correlation** reveals relationships between criteria. High correlations suggest criteria measure similar constructs; low/negative correlations indicate independence.

### Interpretation Guide:
- **|r| < 0.3**: Weak correlation
- **0.3 ≤ |r| < 0.7**: Moderate correlation
- **|r| ≥ 0.7**: Strong correlation

**p-value < 0.05** indicates statistical significance (but requires adequate sample size).

In [ ]:
# Cell 7: Compute Correlation Matrix Between Criteria

def compute_correlation_matrix(scores_df):
    """
    Compute Pearson correlation matrix between criteria.
    
    Returns:
        tuple: (correlation_matrix, p_values_matrix)
    """
    n_criteria = len(scores_df.columns)
    corr_matrix = np.zeros((n_criteria, n_criteria))
    p_matrix = np.zeros((n_criteria, n_criteria))
    
    for i, crit1 in enumerate(scores_df.columns):
        for j, crit2 in enumerate(scores_df.columns):
            if i == j:
                corr_matrix[i, j] = 1.0
                p_matrix[i, j] = 0.0
            else:
                r, p = stats.pearsonr(scores_df[crit1], scores_df[crit2])
                corr_matrix[i, j] = r
                p_matrix[i, j] = p
    
    corr_df = pd.DataFrame(
        corr_matrix,
        index=scores_df.columns,
        columns=scores_df.columns
    )
    
    p_df = pd.DataFrame(
        p_matrix,
        index=scores_df.columns,
        columns=scores_df.columns
    )
    
    return corr_df, p_df

if data:
    corr_matrix, p_matrix = compute_correlation_matrix(scores_df)
    
    print("=" * 80)
    print("CORRELATION ANALYSIS: Criterion Relationships")
    print("=" * 80)
    print("\n⚠ Note: With only", len(scores_df), "systems, correlations have limited statistical power.")
    print("Results should be interpreted as exploratory trends.\n")
    
    print("Pearson Correlation Matrix:")
    print(corr_matrix.round(3))
    
    # Identify strong correlations (|r| > 0.7, excluding diagonal)
    print("\n" + "-" * 80)
    print("Strong Correlations (|r| > 0.7):")
    print("-" * 80)
    
    found_strong = False
    for i, crit1 in enumerate(corr_matrix.index):
        for j, crit2 in enumerate(corr_matrix.columns):
            if i < j:  # Upper triangle only
                r = corr_matrix.iloc[i, j]
                if abs(r) > 0.7:
                    p = p_matrix.iloc[i, j]
                    direction = "positively" if r > 0 else "negatively"
                    print(f"  • {crit1} and {crit2}: r = {r:.3f} (p = {p:.3f})")
                    print(f"    → {direction} correlated")
                    found_strong = True
    
    if not found_strong:
        print("  No strong correlations found (criteria are relatively independent)")

---

## Hierarchical Clustering

**Ward's method** groups systems based on similarity across all criteria. This reveals natural groupings and helps identify which systems are most/least similar.

### Interpretation Guide:
- **Dendrogram height**: Larger distances = more dissimilar systems
- **Cluster membership**: Systems in same cluster have similar profiles
- **Branch order**: Earlier merges = more similar systems

In [ ]:
# Cell 8: Perform Hierarchical Clustering of Systems

def hierarchical_clustering(scores_df, n_clusters=None):
    """
    Perform hierarchical clustering using Ward's method.
    
    Args:
        scores_df: Score matrix
        n_clusters: Number of clusters (None = automatic)
        
    Returns:
        tuple: (linkage_matrix, cluster_labels, distance_matrix)
    """
    # Compute distance matrix (Euclidean distance)
    distance_matrix = pdist(scores_df.values, metric='euclidean')
    
    # Perform hierarchical clustering (Ward's method)
    linkage_matrix = linkage(distance_matrix, method='ward')
    
    # Assign cluster labels
    if n_clusters is None:
        # Use elbow method or default to sqrt(n)
        n_clusters = max(2, int(np.sqrt(len(scores_df))))
    
    cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    
    return linkage_matrix, cluster_labels, squareform(distance_matrix)

if data and len(scores_df) >= 2:
    linkage_mat, clusters, dist_mat = hierarchical_clustering(scores_df)
    
    print("=" * 80)
    print("HIERARCHICAL CLUSTERING: System Groupings")
    print("=" * 80)
    print("\nClustering Method: Ward's linkage (minimizes within-cluster variance)")
    print("Distance Metric: Euclidean distance across all 8 criteria\n")
    
    # Display cluster assignments
    cluster_df = pd.DataFrame({
        'System': scores_df.index,
        'Cluster': clusters
    })
    
    print("Cluster Assignments:")
    for cluster_id in sorted(cluster_df['Cluster'].unique()):
        systems = cluster_df[cluster_df['Cluster'] == cluster_id]['System'].tolist()
        print(f"  Cluster {cluster_id}: {', '.join(systems)}")
    
    # Display distance matrix
    print("\nPairwise Distance Matrix:")
    dist_df = pd.DataFrame(dist_mat, index=scores_df.index, columns=scores_df.index)
    print(dist_df.round(2))
    
    print("\nInterpretation: Lower distances indicate more similar systems.")
else:
    print("⚠ Clustering requires at least 2 systems.")

---

## Visualizations

Publication-ready visualizations generated at 300 DPI for academic papers and presentations.

In [ ]:
# Cell 9: Enhanced Radar Chart (All Systems Overlay)

def create_radar_chart(scores_df, output_path=None):
    """
    Create interactive radar chart with all systems overlaid.
    """
    fig = go.Figure()
    
    # Color palette
    colors = px.colors.qualitative.Set2
    
    for idx, system in enumerate(scores_df.index):
        values = scores_df.loc[system].tolist()
        values.append(values[0])  # Close the polygon
        
        categories = [CRITERIA_DISPLAY[c] for c in scores_df.columns]
        categories.append(categories[0])
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories,
            fill='toself',
            name=system,
            line=dict(color=colors[idx % len(colors)], width=2),
            opacity=0.6
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 5],
                tickmode='linear',
                tick0=0,
                dtick=1
            )
        ),
        showlegend=True,
        title="Comparative Performance Across Evaluation Criteria",
        font=dict(size=12),
        width=800,
        height=600
    )
    
    if output_path:
        fig.write_image(output_path, width=800, height=600, scale=3)  # 300 DPI
        print(f"  ✓ Saved: {output_path}")
    
    return fig

if data:
    print("Generating Enhanced Radar Chart...")
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    radar_path = VIZ_DIR / f'radar_chart_{timestamp}.png'
    radar_fig = create_radar_chart(scores_df, radar_path)
    radar_fig.show()

In [ ]:
# Cell 10: Grouped Bar Chart (Criterion-by-Criterion)

def create_grouped_bar_chart(scores_df, output_path=None):
    """
    Create grouped bar chart for criterion-by-criterion comparison.
    """
    # Reshape data for plotting
    df_melted = scores_df.reset_index().melt(
        id_vars='index',
        var_name='Criterion',
        value_name='Score'
    )
    df_melted.rename(columns={'index': 'System'}, inplace=True)
    df_melted['Criterion'] = df_melted['Criterion'].map(CRITERIA_DISPLAY)
    
    fig = px.bar(
        df_melted,
        x='Criterion',
        y='Score',
        color='System',
        barmode='group',
        title='System Performance by Evaluation Criterion',
        labels={'Score': 'Score (1-5)', 'Criterion': 'Evaluation Criterion'},
        color_discrete_sequence=px.colors.qualitative.Set2,
        height=500
    )
    
    fig.update_layout(
        xaxis_tickangle=-45,
        font=dict(size=11),
        yaxis=dict(range=[0, 5]),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    
    if output_path:
        fig.write_image(output_path, width=1200, height=500, scale=3)
        print(f"  ✓ Saved: {output_path}")
    
    return fig

if data:
    print("Generating Grouped Bar Chart...")
    bar_path = VIZ_DIR / f'bar_chart_{timestamp}.png'
    bar_fig = create_grouped_bar_chart(scores_df, bar_path)
    bar_fig.show()

In [ ]:
# Cell 11: Score Heatmap (Systems × Criteria)

def create_score_heatmap(scores_df, output_path=None):
    """
    Create annotated heatmap of score matrix.
    """
    # Rename columns for display
    display_df = scores_df.copy()
    display_df.columns = [CRITERIA_DISPLAY[c] for c in display_df.columns]
    
    fig = px.imshow(
        display_df,
        labels=dict(x="Evaluation Criterion", y="System", color="Score"),
        x=display_df.columns,
        y=display_df.index,
        color_continuous_scale='RdYlGn',
        aspect='auto',
        title='Score Matrix Heatmap',
        zmin=1,
        zmax=5
    )
    
    # Add text annotations
    fig.update_traces(
        text=display_df.values.round(1),
        texttemplate='%{text}',
        textfont=dict(size=12)
    )
    
    fig.update_layout(
        xaxis_tickangle=-45,
        font=dict(size=11),
        width=1000,
        height=400
    )
    
    if output_path:
        fig.write_image(output_path, width=1000, height=400, scale=3)
        print(f"  ✓ Saved: {output_path}")
    
    return fig

if data:
    print("Generating Score Heatmap...")
    heatmap_path = VIZ_DIR / f'score_heatmap_{timestamp}.png'
    heatmap_fig = create_score_heatmap(scores_df, heatmap_path)
    heatmap_fig.show()

In [ ]:
# Cell 12: Correlation Matrix Heatmap

def create_correlation_heatmap(corr_matrix, p_matrix, output_path=None):
    """
    Create correlation matrix heatmap with significance annotations.
    """
    # Rename for display
    display_corr = corr_matrix.copy()
    display_corr.index = [CRITERIA_DISPLAY[c] for c in display_corr.index]
    display_corr.columns = [CRITERIA_DISPLAY[c] for c in display_corr.columns]
    
    fig = px.imshow(
        display_corr,
        labels=dict(color="Pearson r"),
        x=display_corr.columns,
        y=display_corr.index,
        color_continuous_scale='RdBu_r',
        aspect='auto',
        title='Criterion Correlation Matrix',
        zmin=-1,
        zmax=1
    )
    
    # Add correlation coefficients as annotations
    fig.update_traces(
        text=display_corr.values.round(2),
        texttemplate='%{text}',
        textfont=dict(size=10)
    )
    
    fig.update_layout(
        xaxis_tickangle=-45,
        font=dict(size=10),
        width=900,
        height=800
    )
    
    if output_path:
        fig.write_image(output_path, width=900, height=800, scale=3)
        print(f"  ✓ Saved: {output_path}")
    
    return fig

if data:
    print("Generating Correlation Heatmap...")
    corr_heatmap_path = VIZ_DIR / f'correlation_heatmap_{timestamp}.png'
    corr_fig = create_correlation_heatmap(corr_matrix, p_matrix, corr_heatmap_path)
    corr_fig.show()

In [ ]:
# Cell 13: Dendrogram (Hierarchical Clustering)

def create_dendrogram(linkage_mat, labels, output_path=None):
    """
    Create dendrogram visualization of hierarchical clustering.
    """
    plt.figure(figsize=(10, 6))
    
    dendrogram(
        linkage_mat,
        labels=labels,
        orientation='top',
        distance_sort='descending',
        show_leaf_counts=False,
        leaf_font_size=12
    )
    
    plt.title('Hierarchical Clustering Dendrogram (Ward\'s Method)', fontsize=14, fontweight='bold')
    plt.xlabel('System', fontsize=12)
    plt.ylabel('Distance (Euclidean)', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved: {output_path}")
    
    plt.show()

if data and len(scores_df) >= 2:
    print("Generating Dendrogram...")
    dendro_path = VIZ_DIR / f'dendrogram_{timestamp}.png'
    create_dendrogram(linkage_mat, scores_df.index.tolist(), dendro_path)
else:
    print("⚠ Dendrogram requires at least 2 systems.")

---

## Publication-Ready Exports

In [ ]:
# Cell 14: Export LaTeX Table

def export_latex_table(scores_df, output_path):
    """
    Export score matrix as publication-ready LaTeX table.
    """
    # Rename columns for display
    display_df = scores_df.copy()
    display_df.columns = [CRITERIA_DISPLAY[c] for c in display_df.columns]
    
    # Add mean column
    display_df['Mean'] = display_df.mean(axis=1).round(2)
    
    # Generate LaTeX
    latex_str = display_df.to_latex(
        float_format='%.2f',
        caption='System Performance Scores Across Evaluation Criteria',
        label='tab:system_scores',
        position='htbp',
        column_format='l' + 'c' * len(display_df.columns)
    )
    
    # Add booktabs styling
    latex_str = latex_str.replace('\\toprule', '\\toprule\n\\textbf{System}')
    
    with open(output_path, 'w') as f:
        f.write(latex_str)
    
    print(f"  ✓ Saved: {output_path}")
    return latex_str

if data:
    print("Exporting LaTeX Table...")
    latex_path = SUPP_DIR / f'score_table_{timestamp}.tex'
    latex_table = export_latex_table(scores_df, latex_path)
    print("\nPreview (first 500 chars):")
    print(latex_table[:500] + "...")

In [ ]:
# Cell 15: Generate Markdown Supplementary Materials

def generate_supplementary_markdown(data, scores_df, stats, corr_matrix, output_path):
    """
    Generate comprehensive supplementary materials document.
    """
    md_content = f"""# Supplementary Materials: Comparative System Analysis

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Evaluator:** {data['comparison_metadata']['evaluator']}  
**Systems Compared:** {data['comparison_metadata']['num_systems']}  
**Evaluation Date:** {data['comparison_metadata']['comparison_date']}

---

## 1. System Information

| System | Version | Evaluation Date | Sessions | Journal Entries |
|--------|---------|-----------------|----------|----------------|
"""
    
    for sys in data['systems_compared']:
        md_content += f"| {sys['system_name']} | {sys['system_version']} | {sys['evaluation_date']} | {sys['sessions_imported']} | {sys['journal_entries']} |\n"
    
    md_content += "\n---\n\n## 2. Score Matrix\n\n"
    md_content += scores_df.to_markdown(floatfmt='.2f')
    
    md_content += "\n\n---\n\n## 3. Descriptive Statistics\n\n### Per-Criterion Statistics\n\n"
    md_content += stats['Per-Criterion Statistics'][['mean', 'std', 'min', 'max', 'range']].round(2).to_markdown()
    
    md_content += "\n\n### Per-System Statistics\n\n"
    md_content += stats['Per-System Statistics'][['mean', 'std', 'min', 'max', 'range']].round(2).to_markdown()
    
    md_content += "\n\n---\n\n## 4. Correlation Matrix\n\n"
    md_content += corr_matrix.round(3).to_markdown()
    
    md_content += "\n\n---\n\n## 5. Key Findings\n\n"
    
    # Overall rankings
    overall_means = scores_df.mean(axis=1).sort_values(ascending=False)
    md_content += "### Overall Rankings (by mean score)\n\n"
    for rank, (system, score) in enumerate(overall_means.items(), 1):
        md_content += f"{rank}. **{system}**: {score:.2f}\n"
    
    # Criterion winners
    md_content += "\n### Criterion-Level Winners\n\n"
    for criterion in scores_df.columns:
        winner = scores_df[criterion].idxmax()
        score = scores_df.loc[winner, criterion]
        md_content += f"- **{CRITERIA_DISPLAY[criterion]}**: {winner} ({score:.2f})\n"
    
    md_content += "\n\n---\n\n## 6. Evaluator Insights\n\n"
    
    synthesis = data['evaluator_synthesis']
    md_content += f"**Surprises:** {synthesis['surprises']}\n\n"
    md_content += f"**Confirmations:** {synthesis['confirmations']}\n\n"
    md_content += f"**Open Questions:** {synthesis['open_questions']}\n\n"
    md_content += f"**Evaluator Positionality:** {synthesis['evaluator_positionality']}\n\n"
    md_content += f"**Final Recommendation:** {synthesis['final_recommendation']['single_choice']}\n\n"
    md_content += f"*Rationale:* {synthesis['final_recommendation']['rationale']}\n\n"
    
    md_content += "---\n\n## 7. Visualization Files\n\n"
    md_content += f"All visualizations saved to: `{VIZ_DIR}`\n\n"
    md_content += f"- Radar chart: `radar_chart_{timestamp}.png`\n"
    md_content += f"- Bar chart: `bar_chart_{timestamp}.png`\n"
    md_content += f"- Score heatmap: `score_heatmap_{timestamp}.png`\n"
    md_content += f"- Correlation heatmap: `correlation_heatmap_{timestamp}.png`\n"
    if len(scores_df) >= 2:
        md_content += f"- Dendrogram: `dendrogram_{timestamp}.png`\n"
    
    md_content += "\n---\n\n*Generated by Phase 4 Automated Comparison Utility*\n"
    
    with open(output_path, 'w') as f:
        f.write(md_content)
    
    print(f"  ✓ Saved: {output_path}")
    return md_content

if data:
    print("Generating Supplementary Materials...")
    supp_path = SUPP_DIR / f'supplementary_materials_{timestamp}.md'
    supp_md = generate_supplementary_markdown(data, scores_df, stats, corr_matrix, supp_path)
    print("\n✓ Supplementary materials generated successfully.")

---

## Auto-Generated Insights

In [ ]:
# Cell 16: Generate Statistical Insights

def generate_insights(scores_df, stats, corr_matrix, effect_sizes_df):
    """
    Auto-generate statistical insights from analysis results.
    """
    insights = []
    
    # 1. Overall performance insights
    overall_means = scores_df.mean(axis=1).sort_values(ascending=False)
    best_system = overall_means.index[0]
    best_score = overall_means.iloc[0]
    insights.append(f"Overall Leader: {best_system} (mean: {best_score:.2f})")
    
    # 2. Consistency insights
    system_stds = scores_df.std(axis=1).sort_values()
    most_consistent = system_stds.index[0]
    insights.append(f"Most Consistent: {most_consistent} (std: {system_stds.iloc[0]:.2f})")
    
    # 3. Criterion variability
    criterion_stds = scores_df.std(axis=0).sort_values(ascending=False)
    most_variable = criterion_stds.index[0]
    insights.append(f"Highest System Variation: {CRITERIA_DISPLAY[most_variable]} (std: {criterion_stds.iloc[0]:.2f})")
    
    # 4. Strong correlations
    strong_corrs = []
    for i in range(len(corr_matrix)):
        for j in range(i+1, len(corr_matrix)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                strong_corrs.append((
                    CRITERIA_DISPLAY[corr_matrix.index[i]],
                    CRITERIA_DISPLAY[corr_matrix.columns[j]],
                    corr_matrix.iloc[i, j]
                ))
    
    if strong_corrs:
        for c1, c2, r in strong_corrs:
            insights.append(f"Strong Correlation: {c1} ↔ {c2} (r = {r:.3f})")
    else:
        insights.append("Criterion Independence: No strong correlations (all |r| < 0.7)")
    
    # 5. Largest performance gaps
    largest_gaps = []
    for comparison in effect_sizes_df.index:
        for criterion in effect_sizes_df.columns:
            diff = effect_sizes_df.loc[comparison, criterion]
            if abs(diff) > 1.5:
                largest_gaps.append((comparison, criterion, diff))
    
    if largest_gaps:
        largest_gaps.sort(key=lambda x: abs(x[2]), reverse=True)
        comp, crit, diff = largest_gaps[0]
        sys1, sys2 = comp.split(' vs ')
        direction = "outperforms" if diff > 0 else "underperforms"
        insights.append(f"Largest Gap: {sys1} {direction} {sys2} on {CRITERIA_DISPLAY[crit]} ({abs(diff):.2f} points)")
    
    # 6. Balanced performers
    ranges = scores_df.max(axis=1) - scores_df.min(axis=1)
    most_balanced = ranges.idxmin()
    insights.append(f"Most Balanced: {most_balanced} (range: {ranges.min():.2f})")
    
    # 7. Specialized performers
    for criterion in scores_df.columns:
        winner = scores_df[criterion].idxmax()
        score = scores_df.loc[winner, criterion]
        if score >= 4.5:
            insights.append(f"Excellence: {winner} excels at {CRITERIA_DISPLAY[criterion]} ({score:.2f})")
    
    return insights

if data:
    print("=" * 80)
    print("AUTO-GENERATED INSIGHTS")
    print("=" * 80)
    print("\nStatistical patterns and noteworthy findings:\n")
    
    insights = generate_insights(scores_df, stats, corr_matrix, effect_sizes_df)
    for i, insight in enumerate(insights, 1):
        print(f"{i}. {insight}")
    
    print("\n" + "=" * 80)

---

## Summary and Next Steps

In [ ]:
# Cell 17: Analysis Summary

if data:
    print("=" * 80)
    print("ANALYSIS COMPLETE")
    print("=" * 80)
    print(f"\nAnalysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Source File: {selected_file.name}")
    print(f"Systems Analyzed: {len(scores_df)}")
    print(f"Criteria Evaluated: {len(scores_df.columns)}")
    
    print("\n" + "-" * 80)
    print("OUTPUT FILES GENERATED")
    print("-" * 80)
    
    print("\nVisualizations (300 DPI PNG):")
    print(f"  • Radar chart: {VIZ_DIR / f'radar_chart_{timestamp}.png'}")
    print(f"  • Bar chart: {VIZ_DIR / f'bar_chart_{timestamp}.png'}")
    print(f"  • Score heatmap: {VIZ_DIR / f'score_heatmap_{timestamp}.png'}")
    print(f"  • Correlation heatmap: {VIZ_DIR / f'correlation_heatmap_{timestamp}.png'}")
    if len(scores_df) >= 2:
        print(f"  • Dendrogram: {VIZ_DIR / f'dendrogram_{timestamp}.png'}")
    
    print("\nPublication Materials:")
    print(f"  • LaTeX table: {SUPP_DIR / f'score_table_{timestamp}.tex'}")
    print(f"  • Supplementary materials: {SUPP_DIR / f'supplementary_materials_{timestamp}.md'}")
    
    print("\n" + "-" * 80)
    print("NEXT STEPS")
    print("-" * 80)
    print("\n1. Review visualizations in ../outputs/visualizations/")
    print("2. Incorporate LaTeX table into manuscript (see ../outputs/supplementary/)")
    print("3. Review supplementary materials document for additional context")
    print("4. Consider additional analyses based on auto-generated insights")
    print("5. For publication: Verify all statistical interpretations with domain expert")
    
    print("\n" + "=" * 80)
    print("Thank you for using the Phase 4 Automated Comparison Utility!")
    print("=" * 80)
else:
    print("\n⚠ No data loaded. Please ensure comparative_analysis_*.json file exists.")
    print("\nTo generate input data:")
    print("1. Complete Phase 1 & 2 evaluations for multiple systems")
    print("2. Run Phase 3 comparative dashboard")
    print("3. Export comparative analysis JSON")
    print("4. Re-run this notebook")